# 第4课：Softmax 激活函数

**学习目标：**
- 理解 Softmax 的作用：将任意实数转换为概率分布
- 实现数值稳定的 Softmax
- 将 Softmax 集成到 Network 类

---

神经网络的输出层需要将原始分数（logits）转换为有意义的概率。**Softmax** 就是为此设计的激活函数。

## 4.1 Softmax 的定义

对于向量 $\mathbf{z} = [z_1, z_2, ..., z_n]$：

$$softmax(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{n} e^{z_j}}$$

性质：
- 每个元素 ∈ [0, 1]
- 所有元素之和 = 1
- 可以解释为概率分布

**数值稳定性：** 直接计算 $e^{z_i}$ 可能溢出。减去最大值后再指数运算，结果不变但更安全：

$$softmax(z_i) = softmax(z_i - \max(z))$$

In [ ]:
import numpy as np

def activation_softmax(inputs):
    """Softmax 激活函数（数值稳定版）
    
    参数:
        inputs: shape (m, n)，每行是一个样本的 n 个类别分数
    返回:
        shape (m, n)，每行是概率分布
    """
    max_values = np.max(inputs, axis=1, keepdims=True)  # 每行最大值
    shifted = inputs - max_values        # 减去最大值防溢出
    exp_values = np.exp(shifted)         # 指数
    sum_exp = np.sum(exp_values, axis=1, keepdims=True)  # 每行求和
    return exp_values / sum_exp          # 归一化为概率

## 4.2 Softmax 示例

看看 Softmax 如何将任意实数转换为概率：

In [ ]:
# 2个样本，5个类别
logits = np.random.randn(2, 5)
print("原始分数 (logits):")
print(logits)

probs = activation_softmax(logits)
print("\nSoftmax 概率:")
print(probs)
print("\n每行之和:", np.sum(probs, axis=1))  # 应该都是 1.0

## 4.3 集成到 Network

修改 `Layer` 和 `Network`：
- `Layer.forward` 只计算 $Z = X \cdot W + b$（不加激活）
- `Network.network_forward` 对隐藏层用 ReLU，对输出层用 Softmax

In [ ]:
def activation_ReLU(x):
    return np.maximum(0, x)

class Layer:
    def __init__(self, n_inputs, n_neurons):
        self.weights = np.random.randn(n_inputs, n_neurons)
        self.biases = np.random.randn(n_neurons)

    def forward(self, inputs):
        """只计算线性变换，不加激活"""
        self.sum = np.dot(inputs, self.weights) + self.biases
        return self.sum

class Network:
    def __init__(self, network_shape):
        self.shape = network_shape
        self.layers = []
        for i in range(len(network_shape) - 1):
            self.layers.append(Layer(network_shape[i], network_shape[i + 1]))

    def network_forward(self, inputs):
        outputs = [inputs]
        for i in range(len(self.layers)):
            z = self.layers[i].forward(outputs[i])
            if i == len(self.layers) - 1:
                a = activation_softmax(z)   # 输出层：Softmax
            else:
                a = activation_ReLU(z)      # 隐藏层：ReLU
            outputs.append(a)
        return outputs

In [ ]:
# 测试：3维输入 → 5 → 2维输出
net = Network([3, 5, 2])
inputs = np.array([[1.0, 2.0, 3.0]])
outputs = net.network_forward(inputs)

for i, out in enumerate(outputs):
    print(f"第{i}层输出: {out}")

# 验证输出是概率分布
print(f"\n输出概率之和: {np.sum(outputs[-1]):.6f}")  # 应该 ≈ 1.0

---

## 小结

- Softmax 将任意实数转换为概率分布（和为1）
- 减去最大值保证数值稳定性
- 输出层用 Softmax，隐藏层用 ReLU
- 现在网络可以输出有意义的分类概率了

**下一课**我们将用这个网络做一个真实的分类任务。